In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

GITHUB_REPO = "https://github.com/danokundaye/kidney-tumour-detection.git"
REPO_NAME = "kidney-tumour-detection"
CLONE_PATH = f"/content/{REPO_NAME}"

if os.path.exists(CLONE_PATH):
    print("Repository already exists, pulling latest changes...")
    os.chdir(CLONE_PATH)
    os.system("git pull origin main")
    print("Repository updated")
else:
    print("Cloning repository...")
    os.system(f"git clone {GITHUB_REPO} {CLONE_PATH}")
    print("Repository cloned")

if CLONE_PATH not in sys.path:
    sys.path.insert(0, CLONE_PATH)
    print(f"Added {CLONE_PATH} to Python path")

print(f"\nWorking directory: {CLONE_PATH}")
print("GitHub sync complete")

Mounted at /content/drive
Cloning repository...
Repository cloned
Added /content/kidney-tumour-detection to Python path

Working directory: /content/kidney-tumour-detection
GitHub sync complete


In [2]:
# Verify model weights and test data before pipeline
# checks:
#   1. all three model weight files exist
#   2. test slices directory has 70 cases
#   3. test.csv exists with correct case ids


from pathlib import Path
import pandas as pd

DRIVE_BASE = '/content/drive/MyDrive/kidney-tumour-detection'

models = {
    'yolo'        : 'results/phase5_yolo_retrain/yolov8s_retrain_run1/weights/best.pt',
    'unet'        : 'results/phase6_unet/weights/best.pt',
    'efficientnet': 'results/phase7_efficientnet/weights/best.pt',
}

print('model weights')
for name, rel_path in models.items():
    p = Path(DRIVE_BASE) / rel_path
    if p.exists():
        print(f'  {name}: {p.stat().st_size / 1e6:.1f} mb')
    else:
        print(f'  {name}: NOT FOUND at {p}')

print('\ntest slices')
test_slices = Path(DRIVE_BASE) / 'dataset/processed/slices/test'
cases       = [d.name for d in sorted(test_slices.iterdir()) if d.is_dir()]
print(f'  cases found: {len(cases)}')
print(f'  sample     : {cases[:3]}')

print('\ntest.csv')
test_csv = Path(DRIVE_BASE) / 'dataset/processed/splits/test.csv'
if test_csv.exists():
    df = pd.read_csv(test_csv)
    print(f'  rows      : {len(df)}')
    print(f'  malignant : {df["malignant"].sum()}')
    print(f'  benign    : {(~df["malignant"]).sum()}')
else:
    print('  NOT FOUND')

model weights
  yolo: 22.5 mb
  unet: 130.4 mb
  efficientnet: 48.5 mb

test slices
  cases found: 70
  sample     : ['case_00001', 'case_00003', 'case_00007']

test.csv
  rows      : 70
  malignant : 64
  benign    : 6


In [3]:
import torch
print(torch.cuda.get_device_name(0))

NVIDIA A100-SXM4-40GB


In [5]:
!pip install ultralytics -q
!pip install segmentation-models-pytorch -q
!pip install nibabel -q

import torch
print(f'ultralytics installed')
print(f'segmentation-models-pytorch installed')
print(f'nibabel installed')
print(f'gpu: {torch.cuda.get_device_name(0)}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.3 MB/s eta 0:00:00
ultralytics installed
segmentation-models-pytorch installed
nibabel installed
gpu: NVIDIA A100-SXM4-40GB


In [11]:
# Run end-to-end pipeline on all 70 test cases
# Expected runtime: 20-60 minutes depending on slice count per case
import os
os.chdir("/content/kidney-tumour-detection")
!git pull origin main

!python /content/kidney-tumour-detection/src/pipeline/pipeline.py

From https://github.com/danokundaye/kidney-tumour-detection
 * branch            main       -> FETCH_HEAD
Already up to date.
Phase 9 - End-to-End pipeline inference
started: 2026-02-24 08:52:51

device: cuda
gpu   : NVIDIA A100-SXM4-40GB

Test cases loaded : 70
Output directory  : /content/drive/MyDrive/kidney-tumour-detection/results/phase9_pipeline

Verifying model paths...
  yolo ok: /content/drive/MyDrive/kidney-tumour-detection/results/phase5_yolo_retrain/yolov8s_retrain_run1/weights/best.pt
  unet ok: /content/drive/MyDrive/kidney-tumour-detection/results/phase6_unet/weights/best.pt
  efficientnet ok: /content/drive/MyDrive/kidney-tumour-detection/results/phase7_efficientnet/weights/best.pt

Loading models...
  Loading YOLO from: /content/drive/MyDrive/kidney-tumour-detection/results/phase5_yolo_retrain/yolov8s_retrain_run1/weights/best.pt...
  YOLOv8s loaded
  Loading U-Net (ResNet50) from: /content/drive/MyDrive/kidney-tumour-detection/results/phase6_unet/weights/best.pt...
  

In [12]:
# Pipeline results summary

import pandas as pd
from pathlib import Path

OUTPUT_DIR = '/content/drive/MyDrive/kidney-tumour-detection/results/phase9_pipeline'
pred_csv   = Path(OUTPUT_DIR) / 'predictions.csv'

if not pred_csv.exists():
    print('predictions.csv not found. did cell 2 complete successfully?')
else:
    df = pd.read_csv(pred_csv)

    print('Phase 9 pipeline results\n')
    print(f'cases processed    : {len(df)}')
    print(f'mean dice (3d)     : {df["dice_3d"].mean():.4f}')
    print(f'mean iou (3d)      : {df["iou_3d"].mean():.4f}')
    print(f'mean patches/case  : {df["n_patches"].mean():.1f}')
    print(f'mean time/case (s) : {df["processing_time_s"].mean():.1f}')

    print('\nClassification summary\n')
    print(df['pred_label'].value_counts().to_string())

    print('\nConfidence flags\n')
    print(df['confidence_flag'].value_counts().to_string())

    print('\nPatch method breakdown (totals)\n')
    print(f'  contour      : {df["patches_contour"].sum()}')
    print(f'  small crop   : {df["patches_small"].sum()}')
    print(f'  empty crop   : {df["patches_empty"].sum()}')
    print(f'  no detection : {df["patches_no_det"].sum()}')

    print('\nCases exceeding 120s target\n')
    slow = df[df['processing_time_s'] > 120]
    print(f'  {len(slow)} of {len(df)} cases exceeded 120 seconds')
    if len(slow) > 0:
        print(slow[['case_id', 'n_slices', 'processing_time_s']].to_string(index=False))

Phase 9 pipeline results

cases processed    : 70
mean dice (3d)     : 0.0189
mean iou (3d)      : 0.0123
mean patches/case  : 507.0
mean time/case (s) : 191.3

Classification summary

pred_label
malignant    68
benign        2

Confidence flags

confidence_flag
low_confidence    69
standard           1

Patch method breakdown (totals)

  contour      : 2636
  small crop   : 1038
  empty crop   : 31818
  no detection : 0

Cases exceeding 120s target

  68 of 70 cases exceeded 120 seconds
   case_id  n_slices  processing_time_s
case_00007       512             129.18
case_00055       512             120.87
case_00082       512             131.07
case_00077       512             122.56
case_00204       512             135.69
case_00279       512             132.74
case_00232       512             124.96
case_00096       512             161.78
case_00184       512             186.57
case_00217       512             131.51
case_00264       512             152.73
case_00234       512       